<a href="https://colab.research.google.com/github/JordanDCunha/ngs-underemployment-prediction/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graduate Underemployment Prediction (HackML 2026)

## Objective
Predict whether a graduate is overqualified using survey data.

## Approach
- Data cleaning (handling special codes)
- Feature engineering
- CatBoost with stratified K-Fold cross-validation
- Ensemble strategies (baseline, filtered, weighted)

In [3]:
# ==============================
# Install + Imports
# ==============================
!pip install catboost -q

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from catboost import CatBoostClassifier

## Load Data

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (7709, 25)
Test: (2508, 24)


## Data Cleaning

Replace special survey codes with missing values.

In [5]:
def clean_special_codes(df):
    df = df.copy()
    special_values = [6, 9, 97, 99]

    for col in df.columns:
        df[col] = df[col].replace(special_values, np.nan)

    return df

train = clean_special_codes(train)
test = clean_special_codes(test)

## Feature Engineering

Create additional features to improve model performance.

In [6]:
def feature_engineering(df):
    df = df.copy()

    # Missing indicators
    for col in df.columns:
        if col != "overqualified":
            df[col + "_missing"] = df[col].isna().astype(int)

    # Work experience
    if 'BEF_160' in df.columns:
        df['BEF_160'] = df['BEF_160'].fillna(0)
        df['work_exp_log'] = np.log1p(df['BEF_160'])

    # Education features
    if 'CERTLEVP' in df.columns:
        df['cert_squared'] = df['CERTLEVP'] ** 2

    if 'GRADAGEP' in df.columns and 'CERTLEVP' in df.columns:
        df['age_edu'] = df['GRADAGEP'] * df['CERTLEVP']

    if 'CERTLEVP' in df.columns and 'PREVLEVP' in df.columns:
        df['edu_vs_prev'] = df['CERTLEVP'] - df['PREVLEVP']

    # Parent advantage
    if 'PAR1GRD' in df.columns and 'PAR2GRD' in df.columns:
        df['parent_advantage'] = (
            (df['PAR1GRD'] == 3).astype(int) +
            (df['PAR2GRD'] == 3).astype(int)
        )

    # Financial features
    if 'DBTOTGRD' in df.columns and 'SCHOLARP' in df.columns:
        df['financial_pressure'] = df['DBTOTGRD'] - df['SCHOLARP']
        df['loan_scholar_ratio'] = df['DBTOTGRD'] / (df['SCHOLARP'] + 1)

    return df

train = feature_engineering(train)
test = feature_engineering(test)

## Prepare Data

In [7]:
target = "overqualified"

X = train.drop(columns=[target, "id"]).astype(str)
y = train[target]

X_test = test.drop(columns=["id"])

## Model Training (Stratified K-Fold)

In [8]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_preds = np.zeros(len(X_test))
filtered_preds = np.zeros(len(X_test))
weighted_preds = np.zeros(len(X_test))

val_scores = []
weights = []
good_folds = 0

In [10]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n🔥 Fold {fold+1}")

    # ✅ Convert to string to avoid CatBoost errors
    X_train = X.iloc[train_idx].astype(str)
    X_val = X.iloc[val_idx].astype(str)

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=list(range(X.shape[1])),
        use_best_model=True
    )

    val_preds = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, val_preds)

    print("Fold AUC:", score)

    # ✅ ALSO ensure test is string
    test_fold_preds = model.predict_proba(X_test.astype(str))[:, 1]

    # ==============================
    # Baseline
    # ==============================
    baseline_preds += test_fold_preds / 5

    # ==============================
    # Filtered
    # ==============================
    if score > 0.70:
        print("✅ Using for filtered")
        filtered_preds += test_fold_preds
        good_folds += 1

    # ==============================
    # Weighted
    # ==============================
    weights.append(score)
    weighted_preds += test_fold_preds * score

    val_scores.append(score)


🔥 Fold 1
Fold AUC: 0.7159786550855876
✅ Using for filtered

🔥 Fold 2
Fold AUC: 0.6918598969004384

🔥 Fold 3
Fold AUC: 0.6939940329485018

🔥 Fold 4
Fold AUC: 0.7026682919981448
✅ Using for filtered

🔥 Fold 5
Fold AUC: 0.6859834676615499


## Results & Submission

In [11]:
filtered_preds = filtered_preds / max(good_folds, 1)
weighted_preds = weighted_preds / sum(weights)

print("\n🔥 Mean CV AUC:", np.mean(val_scores))
print("🔥 Good folds used:", good_folds)

# Save submissions
pd.DataFrame({
    "id": test["id"],
    "overqualified": baseline_preds
}).to_csv("submission_baseline.csv", index=False)

pd.DataFrame({
    "id": test["id"],
    "overqualified": filtered_preds
}).to_csv("submission_filtered.csv", index=False)

pd.DataFrame({
    "id": test["id"],
    "overqualified": weighted_preds
}).to_csv("submission_weighted.csv", index=False)

print("✅ All submissions saved!")


🔥 Mean CV AUC: 0.6980968689188446
🔥 Good folds used: 2
✅ All submissions saved!
